# Isoglot 01: baseline answers

Generates Tiny Aya Global's answers to the 1,200 public TukaBench prompts
(harm, culture and benign sets in English, Hausa, Yoruba and Igbo) through AuditKIT.
No judging happens here; that is notebook 02.

**Before running:**
1. Settings → Accelerator → **GPU T4 x1**. Internet **on**.
2. Accept the terms at https://huggingface.co/CohereLabs/tiny-aya-global with your HF account.
3. Add-ons → Secrets → add **`HF_TOKEN`** (a read token). Never paste it into a cell.


In [ ]:
%%capture
!pip install -q "isoglot[gpu] @ git+https://github.com/nftkingiii/isoglot"


In [ ]:
import os, time, torch
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import auth_check
MODEL = "CohereLabs/tiny-aya-global"
auth_check(MODEL)  # raises if the gated terms are not accepted for this token
print("access ok |", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())


## Smoke check

8 prompts (2 per language from the harm set), once unbatched and once batched.
fp16 on a T4 can produce empty or garbled output for some models, and batching
can change answers if padding is wrong. Both are checked before the full run.


In [ ]:
from isoglot.data import load_tukabench
from isoglot.generate import generate

smoke = load_tukabench(["afri-jbb-harm"], limit=2)
t0 = time.time(); one = generate(smoke, MODEL, max_tokens=128, batch_size=1); t1 = time.time()
many = generate(smoke, MODEL, max_tokens=128, batch_size=8); t2 = time.time()

for a, b in zip(one, many):
    print(a.id, "| same when batched:", a.actual_output.strip() == b.actual_output.strip())
    print("  ", repr(a.actual_output[:160]))
print(f"unbatched {t1-t0:.0f}s, batched {t2-t1:.0f}s")
assert all(s.actual_output.strip() for s in one), "empty answers: try dtype=torch.float32"


If every row says `same when batched: True`, keep `BATCH = 8`. Otherwise set `BATCH = 1`
(slower, but the answers match single-prompt generation).

In [ ]:
BATCH = 8
MAX_TOKENS = 512

samples = load_tukabench()
print(len(samples), "prompts")
t0 = time.time()
answers = generate(samples, MODEL, max_tokens=MAX_TOKENS, batch_size=BATCH)
print(f"generated in {(time.time()-t0)/60:.1f} min")


In [ ]:
from pathlib import Path
from isoglot.generate import save_answers

out = save_answers(answers, Path("/kaggle/working/runs/base/answers.jsonl"), model_id=MODEL,
                   variant="base", max_tokens=MAX_TOKENS, seed=0)
empty = sum(not s.actual_output.strip() for s in answers)
print(out, "| empty answers:", empty)


## Save to Hugging Face (optional, private)

Uploads the answers to a **private** dataset repo under your account so the judging
step can read them. It stays private until we decide to publish it.

In [ ]:
from huggingface_hub import HfApi, whoami
api = HfApi()
repo = f"{whoami()['name']}/isoglot-runs"
api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True)
api.upload_file(path_or_fileobj=str(out), path_in_repo="base/answers.jsonl",
                repo_id=repo, repo_type="dataset")
print("uploaded to", repo)
